<a href="https://colab.research.google.com/github/CVaruni/Real-Estate-Analyzer/blob/main/notebooks/Real_Estate_Rent_vs_Buy_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install plotly pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [4]:
rent_df = pd.read_csv('allcities cleaned.csv')
buy_df = pd.read_csv('real estate data v2.csv')

print("RENT DATA:")
print(rent_df.shape)
print(rent_df.columns.tolist())
print(rent_df.head(3))

print("\nBUY DATA:")
print(buy_df.shape)
print(buy_df.columns.tolist())
print(buy_df.head(3))

RENT DATA:
(193011, 10)
['seller_type', 'bedroom', 'layout_type', 'property_type', 'locality', 'price', 'area', 'furnish_type', 'bathroom', 'city']
  seller_type  bedroom layout_type     property_type  locality    price  \
0       OWNER      2.0         BHK         Apartment  Bodakdev  20000.0   
1       OWNER      1.0          RK  Studio Apartment   CG Road   7350.0   
2       OWNER      3.0         BHK         Apartment   Jodhpur  22000.0   

     area    furnish_type  bathroom       city  
0  1450.0       Furnished       2.0  Ahmedabad  
1   210.0  Semi-Furnished       1.0  Ahmedabad  
2  1900.0     Unfurnished       3.0  Ahmedabad  

BUY DATA:
(14528, 9)
['Name', 'Property Title', 'Price', 'Location', 'Total_Area', 'Price_per_SQFT', 'Description', 'Baths', 'Balcony']
                                    Name  \
0                       Casagrand ECR 14   
1  Ramanathan Nagar, Pozhichalur,Chennai   
2                            DAC Prapthi   

                                      Pro

In [5]:
# Drop nulls
rent_df = rent_df.dropna(subset=['price', 'area', 'city', 'bedroom'])

# Standardize city names
rent_df['city'] = rent_df['city'].str.strip().str.title()

# Filter outliers
rent_df = rent_df[rent_df['price'] < 500000]
rent_df = rent_df[rent_df['area'] < 10000]

# Add price per sqft
rent_df['price_per_sqft'] = rent_df['price'] / rent_df['area']

print("✅ Rent data cleaned!")
print(rent_df['city'].value_counts())
print(rent_df.shape)

✅ Rent data cleaned!
city
Mumbai       67215
Delhi        31258
Bangalore    23065
Pune         22594
Ahmedabad    18452
Hyderabad    10537
Kolkata       9656
Chennai       8650
Name: count, dtype: int64
(191427, 11)


In [6]:
# Clean price column - convert ₹1.99 Cr to numeric
def convert_price(price_str):
    try:
        price_str = str(price_str).replace('₹', '').replace(',', '').strip()
        if 'Cr' in price_str:
            return float(price_str.replace('Cr', '').strip()) * 10000000
        elif 'L' in price_str or 'Lac' in price_str:
            return float(price_str.replace('L', '').replace('Lac', '').strip()) * 100000
        else:
            return float(price_str)
    except:
        return np.nan

buy_df['price_numeric'] = buy_df['Price'].apply(convert_price)

# Extract city from Location
buy_df['city'] = buy_df['Location'].str.split(',').str[-1].str.strip().str.title()

# Drop nulls
buy_df = buy_df.dropna(subset=['price_numeric', 'Total_Area', 'Price_per_SQFT'])

# Filter valid rows
buy_df = buy_df[buy_df['price_numeric'] > 0]
buy_df = buy_df[buy_df['Total_Area'] > 0]

print("✅ Buy data cleaned!")
print(buy_df['city'].value_counts())
print(buy_df.shape)

✅ Buy data cleaned!
city
Bangalore    4512
Pune         2964
New Delhi    2164
Chennai      1595
Kolkata      1392
Mumbai       1353
Hyderabad     540
Thane           6
Name: count, dtype: int64
(14526, 11)


In [7]:
conn = sqlite3.connect('real_estate.db')

rent_df.to_sql('rent', conn, if_exists='replace', index=False)
buy_df.to_sql('buy', conn, if_exists='replace', index=False)

# Verify
print("✅ Data stored in SQL!")
print(pd.read_sql("SELECT city, COUNT(*) as listings FROM rent GROUP BY city", conn))

✅ Data stored in SQL!
        city  listings
0  Ahmedabad     18452
1  Bangalore     23065
2    Chennai      8650
3      Delhi     31258
4  Hyderabad     10537
5    Kolkata      9656
6     Mumbai     67215
7       Pune     22594


In [8]:
avg_rent = rent_df.groupby('city')['price'].mean().sort_values(ascending=False).reset_index()
avg_rent.columns = ['city', 'avg_rent']

fig = px.bar(avg_rent, x='city', y='avg_rent',
             color='avg_rent',
             color_continuous_scale='Reds',
             title='💰 Average Monthly Rent by City',
             labels={'avg_rent': 'Avg Rent (₹)', 'city': 'City'})
fig.update_layout(template='plotly_dark')
fig.show()

In [9]:
avg_buy = buy_df.groupby('city')['price_numeric'].mean().sort_values(ascending=False).reset_index()
avg_buy.columns = ['city', 'avg_buy_price']

fig = px.bar(avg_buy, x='city', y='avg_buy_price',
             color='avg_buy_price',
             color_continuous_scale='Blues',
             title='🏠 Average Property Buy Price by City',
             labels={'avg_buy_price': 'Avg Buy Price (₹)', 'city': 'City'})
fig.update_layout(template='plotly_dark')
fig.show()

In [10]:
avg_rent_sqft = rent_df.groupby('city')['price_per_sqft'].mean().reset_index()
avg_rent_sqft.columns = ['city', 'rent_per_sqft']

avg_buy_sqft = buy_df.groupby('city')['Price_per_SQFT'].mean().reset_index()
avg_buy_sqft.columns = ['city', 'buy_per_sqft']

# Merge
comparison_df = pd.merge(avg_rent_sqft, avg_buy_sqft, on='city')

fig = go.Figure()
fig.add_trace(go.Bar(name='Rent/sqft (₹/month)', x=comparison_df['city'], y=comparison_df['rent_per_sqft'], marker_color='tomato'))
fig.add_trace(go.Bar(name='Buy/sqft (₹)', x=comparison_df['city'], y=comparison_df['buy_per_sqft'], marker_color='steelblue'))

fig.update_layout(barmode='group',
                  title='📊 Rent vs Buy — Price per SQFT by City',
                  template='plotly_dark',
                  yaxis_title='Price (₹)')
fig.show()

In [11]:
# How many months of rent = buying a property?
comparison_df['breakeven_months'] = comparison_df['buy_per_sqft'] / comparison_df['rent_per_sqft']
comparison_df['breakeven_years'] = (comparison_df['breakeven_months'] / 12).round(1)
comparison_df['verdict'] = comparison_df['breakeven_years'].apply(
    lambda x: '🔴 Buy Expensive' if x > 20 else ('🟡 Moderate' if x > 10 else '🟢 Worth Buying')
)

fig = px.bar(comparison_df, x='city', y='breakeven_years',
             color='breakeven_years',
             color_continuous_scale='RdYlGn_r',
             title='⏳ Breakeven Analysis — Years of Rent to Equal Buying Cost',
             labels={'breakeven_years': 'Years', 'city': 'City'},
             text='breakeven_years')
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_dark')
fig.show()

print("\n📋 City Verdicts:")
print(comparison_df[['city', 'breakeven_years', 'verdict']].to_string(index=False))


📋 City Verdicts:
     city  breakeven_years         verdict
Bangalore             50.2 🔴 Buy Expensive
  Chennai             37.1 🔴 Buy Expensive
Hyderabad             40.3 🔴 Buy Expensive
  Kolkata             43.4 🔴 Buy Expensive
   Mumbai             42.2 🔴 Buy Expensive
     Pune             39.7 🔴 Buy Expensive


In [14]:
fig = px.box(rent_df, x='city', y='price',
             color='city',
             title='📦 Rent Price Distribution by City',
             labels={'price': 'Monthly Rent (₹)', 'city': 'City'},
             template='plotly_dark')
fig.update_layout(showlegend=False)
fig.update_yaxes(range=[0, 100000])
fig.show()

In [15]:
bhk_rent = rent_df.groupby(['city', 'bedroom'])['price'].mean().reset_index()
bhk_rent = bhk_rent[bhk_rent['bedroom'].isin([1, 2, 3])]

fig = px.bar(bhk_rent, x='city', y='price', color='bedroom',
             barmode='group',
             title='🛏️ Average Rent by BHK Type per City',
             labels={'price': 'Avg Rent (₹)', 'bedroom': 'BHK'},
             template='plotly_dark',
             color_continuous_scale='Viridis')
fig.show()

In [16]:
furnish_rent = rent_df.groupby(['city', 'furnish_type'])['price'].mean().reset_index()

fig = px.bar(furnish_rent, x='city', y='price', color='furnish_type',
             barmode='group',
             title='🛋️ Impact of Furnishing on Rent by City',
             labels={'price': 'Avg Rent (₹)', 'furnish_type': 'Furnishing Type'},
             template='plotly_dark')
fig.show()

In [17]:
print("=" * 55)
print("   🏙️ INDIAN REAL ESTATE — RENT VS BUY INSIGHTS")
print("=" * 55)

print("\n📌 BREAKEVEN ANALYSIS (Years of rent = buying cost):")
for _, row in comparison_df.sort_values('breakeven_years').iterrows():
    print(f"  {row['verdict']}  {row['city']:<12} → {row['breakeven_years']} years")

print("\n📌 MOST AFFORDABLE CITY TO RENT:")
cheapest = rent_df.groupby('city')['price'].mean().idxmin()
print(f"  🟢 {cheapest}")

print("\n📌 MOST EXPENSIVE CITY TO RENT:")
priciest = rent_df.groupby('city')['price'].mean().idxmax()
print(f"  🔴 {priciest}")

print("\n📌 MOST EXPENSIVE CITY TO BUY:")
priciest_buy = buy_df.groupby('city')['price_numeric'].mean().idxmax()
print(f"  🔴 {priciest_buy}")

print("\n📌 OVERALL VERDICT:")
print("  🏠 Renting is financially smarter across ALL")
print("  major Indian metros. Minimum breakeven is")
avg_min = comparison_df['breakeven_years'].min()
print(f"  {avg_min} years — making buying a long-term")
print("  commitment rather than a short-term gain.")
print("=" * 55)

   🏙️ INDIAN REAL ESTATE — RENT VS BUY INSIGHTS

📌 BREAKEVEN ANALYSIS (Years of rent = buying cost):
  🔴 Buy Expensive  Chennai      → 37.1 years
  🔴 Buy Expensive  Pune         → 39.7 years
  🔴 Buy Expensive  Hyderabad    → 40.3 years
  🔴 Buy Expensive  Mumbai       → 42.2 years
  🔴 Buy Expensive  Kolkata      → 43.4 years
  🔴 Buy Expensive  Bangalore    → 50.2 years

📌 MOST AFFORDABLE CITY TO RENT:
  🟢 Hyderabad

📌 MOST EXPENSIVE CITY TO RENT:
  🔴 Delhi

📌 MOST EXPENSIVE CITY TO BUY:
  🔴 Mumbai

📌 OVERALL VERDICT:
  🏠 Renting is financially smarter across ALL
  major Indian metros. Minimum breakeven is
  37.1 years — making buying a long-term
  commitment rather than a short-term gain.


In [18]:
comparison_df.to_csv('rent_vs_buy_summary.csv', index=False)
rent_df.to_csv('rent_cleaned.csv', index=False)
print("✅ Files saved!")
print(comparison_df)

✅ Files saved!
        city  rent_per_sqft  buy_per_sqft  breakeven_months  breakeven_years  \
0  Bangalore      18.928432  11396.265514        602.071299             50.2   
1    Chennai      18.268838   8132.200627        445.140546             37.1   
2  Hyderabad      17.006777   8215.833333        483.091718             40.3   
3    Kolkata      17.445212   9089.525862        521.032689             43.4   
4     Mumbai      41.233617  20874.833703        506.257647             42.2   
5       Pune      19.237873   9157.844130        476.032055             39.7   

           verdict  
0  🔴 Buy Expensive  
1  🔴 Buy Expensive  
2  🔴 Buy Expensive  
3  🔴 Buy Expensive  
4  🔴 Buy Expensive  
5  🔴 Buy Expensive  
